# RLPT-00f — Le SOTA de l'alignement en ligne : `trl.GRPOTrainer` contre une boucle PPO maison

**Série** Post-Training RL (`rlpt_*`) — niveau intermédiaire · **Prérequis** : [`RLPT-00`](RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb) (le monde synthétique et le reward model Bradley-Terry, repris verbatim), [`RLPT-01`](RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb) (PPO from scratch), [`RLPT-00e`](RLPT-00e-DPO-TRL-Python.ipynb) (le geste SOTA côté offline, même monde).

Ce notebook ferme l'arc ouvert par [`RLPT-00e`](RLPT-00e-DPO-TRL-Python.ipynb) : là où DPO apprenait
**offline** (un dataset figé de préférences), ici la politique **échantillonne ses propres
réponses** et apprend en ligne du score d'un juge — l'arc complet RLHF de `RLPT-01`.
Le geste SOTA est le même : brancher la lib industrielle sur notre monde et départager
à budget égal.

Un fait d'abord, mesuré dans la cellule d'environnement : **`trl.PPOTrainer` a quitté la racine
de la lib, et l'espace `trl.experimental.ppo` qui l'avait reçu n'est plus livré non plus**.
L'issue #16063 (bloc B.6) demandait « `trl.PPOTrainer` sur la même mini-tâche que RLPT-01 » : la
surface a existé de `trl` 1.10.0 à 1.12.0, puis elle a été **retirée du paquet en `trl` 1.13.0**
(fenêtre mesurée par inspection des wheels PyPI). Ce notebook compare donc ce qui s'exécute
encore aujourd'hui, et **constate** l'absence de ce qui ne s'exécute plus :

| Bras | Algorithme | Baseline de l'avantage | Pile |
|---|---|---|---|
| **Maison** | PPO-clip (recette `RLPT-01` transplantée) | value net **appris** $V(x)$ | torch pur |
| **SOTA** | GRPO (group relative policy optimization) | **moyenne du groupe** de $G$ réponses | `trl` |
| **Historique** | PPO complet (critic + GAE, API Trainer) | value head **appris** $V(s_t)$ + GAE $\lambda$ | `trl.experimental.ppo` — **retirée du paquet en 1.13.0**, non exécutée ici |

Maison et GRPO résolvent le même problème — *réduire la variance d'une récompense terminale
unique* — par deux baselines différentes. C'est LA différence pédagogique du notebook :
un critic entraîné, ou la structure du groupe elle-même. Le troisième bras reste dans le
carnet **en lecture** : son code (le contrat d'API `reward_model` au format de la lib) est
conservé tel quel et s'exécute chez qui installe `trl==1.12.0` ; sous la version installée
ici, son absence est **affichée** dans la sortie, jamais subie en silence.


In [1]:
import os, sys, time, tempfile, platform
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import transformers, trl, datasets
from scipy.stats import spearmanr

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__}"
      f" | trl {trl.__version__} | datasets {datasets.__version__}")

# Le fait structurant : PPOTrainer a quitte la RACINE de trl, pas la lib
exports = [x for x in dir(trl) if 'Trainer' in x]
print("trainers trl (racine) :", ', '.join(sorted(exports)))
print("PPOTrainer a la racine :", hasattr(trl, 'PPOTrainer'))
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")   # le module est declare instable, on sait
# La surface ou la classe vivait : elle est RETIREE du paquet TRL en 1.13.0 (fenetre mesuree :
# presente dans les wheels 1.10.0, 1.11.0 et 1.12.0, absente a partir de 1.13.0). On CONSTATE
# l'etat de la surface installee -- on ne suppose plus qu'elle existe.
try:
    from trl.experimental.ppo import PPOTrainer, PPOConfig
    _PPO_AVAILABLE = True
    print("PPOTrainer (reel) :", PPOTrainer.__module__)
except ModuleNotFoundError as exc:
    _PPO_AVAILABLE = False
    PPOTrainer = PPOConfig = None
    print("PPOTrainer (reel) : ABSENT --", exc)
print("surface trl.experimental.ppo disponible :", _PPO_AVAILABLE)


python 3.13.16 | torch 2.9.0+cu126 | transformers 5.2.0 | trl 1.17.1 | datasets 5.1.0
trainers trl (racine) : DPOTrainer, DistillationTrainer, GRPOTrainer, KTOTrainer, RLOOTrainer, RewardTrainer, SFTTrainer
PPOTrainer a la racine : False
PPOTrainer (reel) : ABSENT -- No module named 'trl.experimental.ppo'
surface trl.experimental.ppo disponible : False


**Lecture chiffrée — le socle de versions et le fait qui organise le notebook.** `python 3.13.16 | torch 2.9.0+cu126 | transformers 5.2.0 | trl 1.17.1 | datasets 5.1.0`, puis l'inventaire : `trainers trl (racine) : DPOTrainer, DistillationTrainer, GRPOTrainer, KTOTrainer, RLOOTrainer, RewardTrainer, SFTTrainer` et surtout la paire `PPOTrainer a la racine : False` / `surface trl.experimental.ppo disponible : False`. Ces deux lignes expliquent l'architecture entière du comparatif : `trl.PPOTrainer` a quitté la racine de la lib (déprécié fin 2024), et l'espace `trl.experimental.ppo` qui l'accueillait a été **retiré du paquet en `trl` 1.13.0** — la fenêtre de disponibilité mesurée va de `trl` 1.10.0 à 1.12.0. La comparaison honnête possible aujourd'hui se joue donc en deux temps exécutés : une boucle PPO écrite à la main (section 4) et le `trl.GRPOTrainer` officiel (section 5), sur le même monde et le même budget de rollouts ; la troisième branche (section 5bis) reste **lisible mais non exécutée**, et son absence est affichée par la cellule elle-même. Tout ce qui suit (bras maison, bras SOTA, multi-seed, verdict) découle de ces deux lignes d'inventaire.


`trl.PPOTrainer` n'est plus exposé à la **racine** de la lib (déprécié fin 2024 ; la racine
ne porte plus que les traineurs en ligne « modernes » : `GRPOTrainer`, `RLOOTrainer`), et
l'espace `trl.experimental.ppo` qui l'avait reçu n'est **plus livré** : la surface a été
retirée du paquet en `trl` 1.13.0, après une fenêtre de disponibilité de 1.10.0 à 1.12.0.
Le « PPO-RLHF SOTA comparison » de l'issue se livre donc en deux temps exécutés — la boucle
maison et le `GRPOTrainer` que la pratique industrielle a adopté (c'est le cœur de `RLPT-02`
côté main, et des notebooks `PT-11a/b` côté échelle réelle) — et en un troisième bras
**conservé en lecture** (section 5bis) : le code du `PPOTrainer` historique y est toujours
lisible, mais sous la version de `trl` installée ici il ne s'exécute plus.

**Ce que cet usage enseigne, et que le carnet garde volontairement.** Une « surface SOTA »
n'est pas un fait permanent : c'est un fait **daté par la version de la bibliothèque**. Le
même carnet, exécuté sous `trl` 1.9.2, produisait trois colonnes ; sous `trl` 1.17.1, il
en produit deux et **constate** la troisième comme absente. Une comparaison SOTA se date
donc deux fois — par le noyau, et par la bibliothèque.

**GRPO en une phrase** : au lieu d'entraîner un critic $V(s)$ à prédire la récompense,
on génère $G$ réponses au même prompt, et l'avantage de chacune est
$A_i = (r_i - \text{mean}(r_{1..G})) / \text{std}(r_{1..G})$ — le groupe est sa propre
baseline. Pas de value net, pas de GAE : la variance se réduit par la structure même de
l'échantillonnage.


## 1. Le monde synthétique (verbatim `RLPT-00`)

Même monde que `RLPT-00`/`RLPT-00e` : vocabulaire de 8 lettres `a..h` à poids connus,
deux prompts `<pA>`/`<pB>` avec bonus positionnels conditionnels, réponses de 8 tokens.
La **vraie récompense** $r^*$ sert uniquement de **métrique** — l'entraînement, lui,
ne verra jamais que le score du juge appris (section 2).

In [2]:
TOK = ['<pA>', '<pB>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', '<eos>']
V = {c: i for i, c in enumerate(TOK)}
LEN_R = 8
W_TOK = {'a': 0.30, 'b': 0.05, 'c': 0.10, 'd': 0.15, 'e': 0.25, 'f': -0.10, 'g': -0.15, 'h': -0.20}
SEEDS = [0, 1, 7, 42]

def true_reward(seq, prompt):
    r = sum(W_TOK[TOK[t]] for t in seq)
    if prompt == 0 and seq[0] == V['a']:
        r += 0.8
    if prompt == 1 and seq[4] == V['e']:
        r += 0.8
    return r

def sample_response(rng):
    return rng.integers(2, len(TOK) - 1, size=LEN_R)   # contenu seulement

def make_pairs(n_pairs, rng, min_gap=0.3, beta=1.0):
    pw, y, pr = [], [], []
    while len(pw) < n_pairs:
        prompt = int(rng.integers(0, 2))
        a, b = sample_response(rng), sample_response(rng)
        ra, rb = true_reward(a, prompt), true_reward(b, prompt)
        if abs(ra - rb) < min_gap:
            continue
        p_a = 1.0 / (1.0 + np.exp(-beta * (ra - rb)))
        pw.append((a, b)); y.append(1 if rng.random() < p_a else 0); pr.append(prompt)
    return pw, np.array(y), np.array(pr)

rng0 = np.random.default_rng(7)
fond = np.mean([true_reward(sample_response(rng0), int(rng0.integers(0, 2))) for _ in range(4000)])
print(f"bruit de fond (politique uniforme sur le contenu) : {fond:.3f}")
# Le plafond se calcule par la MEME fonction que la vraie recompense : la regle de bonus
# differe entre les deux prompts (seq[0]=='a' pour <pA>, seq[4]=='e' pour <pB>).
greedy_a = true_reward([V['a']]*8, 0)
greedy_b = true_reward([V['a']]*4 + [V['e']]*4, 1)
print(f"plafond glouton : <pA> -> a*8 + bonus = {greedy_a:.2f} | <pB> -> a*4+e*4 + bonus = {greedy_b:.2f}")


bruit de fond (politique uniforme sur le contenu) : 0.487
plafond glouton : <pA> -> a*8 + bonus = 3.20 | <pB> -> a*4+e*4 + bonus = 3.00


**Lecture chiffrée — les repères pour lire toutes les courbes.** `bruit de fond (politique uniforme sur le contenu) : 0.487` et `plafond glouton : <pA> -> a*8 + bonus = 3.20 | <pB> -> a*4+e*4 + bonus = 3.00`. Toute récompense de ce notebook se lit entre ces bornes : 0.487 = ce qu'une politique qui tire les lettres au hasard marque en moyenne (le bonus positionnel tombe rarement tout seul), 3.20 et 3.00 = le maximum atteignable en répondant `a` à chaque position pour `<pA>`, et `a` quatre fois puis `e` quatre fois pour `<pB>`. Les deux plafonds sont calculés **par la fonction de vraie récompense** (`true_reward`), donc ils appliquent la règle de bonus de chaque prompt : celle de `<pB>` se déclenche sur le cinquième token, pas sur le premier. Une formule figée qui l'ignore sous-estime le plafond de `<pB>` de 0.60 — c'est le genre d'écart qui fait lire « 2.40 » à un lecteur qui devrait lire « 3.00 ». Le 0.487 est une moyenne empirique sur 4 000 réponses tirées avec la graine 7, pas une espérance analytique : une nouvelle graine peut déplacer ce repère. Les deux plafonds correspondent à des prompts différents et ne sont donc pas deux scores concurrents pour une même réponse. Les gains mesurés plus bas prennent leur sens contre le bruit de fond 0.487 : passer d'une récompense sous le hasard à une récompense au-dessus, c'est traverser la ligne du hasard, pas toucher le plafond.


Deux repères pour lire toutes les courbes de ce notebook : une politique uniforme sur
les lettres marque 0.487 en moyenne sur les 4 000 tirages de la cellule précédente
(le bonus positionnel tombe rarement), et la meilleure réponse possible vaut
3.20 (`<pA>`) / 3.00 (`<pB>`). Une politique à 0,8-0,9 dépasse donc le bruit
de fond — sans pour autant épuiser le monde.


## 2. Le juge : reward model Bradley-Terry (recette `RLPT-00`, compressée)

L'arc RLHF complet exige un **reward model appris** — pas la vraie récompense. On
réentraîne le RM de `RLPT-00` : paires de préférences bruitées par un Bradley-Terry,
MLE par paires, ~5 s. **Les deux bras (PPO maison et GRPO) optimisent le score de CE
RM** — le même juge, les mêmes données. La vraie récompense n'intervient qu'aux
évaluations : c'est là que se mesurera l'écart juge/vrai (la faille de tout RLHF).

In [3]:
def encode(pair_list, prompt_list):
    B = len(pair_list)
    x = torch.zeros(B, 2, 1 + LEN_R, dtype=torch.long)
    for i, (a, b) in enumerate(pair_list):
        x[i, 0, 0] = x[i, 1, 0] = prompt_list[i]
        x[i, 0, 1:] = torch.from_numpy(np.asarray(a, dtype=np.int64))
        x[i, 1, 1:] = torch.from_numpy(np.asarray(b, dtype=np.int64))
    return x

class RewardModel(nn.Module):
    def __init__(self, vs=len(TOK), hid=64):
        super().__init__()
        self.emb = nn.Embedding(vs, hid)
        self.pos = nn.Embedding(1 + LEN_R, hid)
        self.mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, x):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(L))
        return self.mlp(h.mean(1)).squeeze(-1)

def train_bt(model, x, y, epochs=60, bs=256, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(y)
    for ep in range(epochs):
        perm = torch.randperm(n).tolist()
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            xb, yb = x[idx], torch.tensor(y[idx], dtype=torch.long)
            b = len(idx)
            win = xb[torch.arange(b), 1 - yb]
            lose = xb[torch.arange(b), yb]
            loss = F.binary_cross_entropy_with_logits(model(win) - model(lose), torch.ones(b))
            opt.zero_grad(); loss.backward(); opt.step()

def make_rm(seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pw, y, pr = make_pairs(4000, rng)
    rm = RewardModel()
    train_bt(rm, encode(pw, pr), y)
    return rm

t0 = time.time()
RM = make_rm(0)
print(f"RM (seed 0) entraîné en {time.time()-t0:.1f}s -- c'est le juge des deux bras")

RM (seed 0) entraîné en 12.5s -- c'est le juge des deux bras


## 3. La politique, la métrique, et le garde-fou d'évaluation

Le LM est celui de `RLPT-00e` : GPT-2 de jouet (vocab 11, 2 couches, 64 dim, ~100k
paramètres). La **métrique** est commune aux deux bras : on échantillonne la politique
(sans masque — elle peut émettre n'importe quoi), on filtre les tokens de contenu
(`a..h`), on complète avec `h` (le pire poids) si elle bavarde du prompt/EOS — la
dégénérescence est punie, pas récompensée — et on score à la **vraie** récompense.

In [4]:
from transformers import AutoModelForCausalLM, GPT2Config, PreTrainedTokenizerFast
from tokenizers import Tokenizer, models, pre_tokenizers

def make_tz():
    tm = models.WordLevel(vocab={c: i for i, c in enumerate(TOK)}, unk_token='<eos>')
    tk = Tokenizer(tm)
    tk.pre_tokenizer = pre_tokenizers.WhitespaceSplit()   # pas Whitespace (regex qui déchire <pB>)
    return PreTrainedTokenizerFast(tokenizer_object=tk, pad_token='<pA>', bos_token='<pA>', eos_token='<eos>')

tz = make_tz()
assert tz('<pB> a b c d e f g h')["input_ids"] == [1, 2, 3, 4, 5, 6, 7, 8, 9]

def make_lm(seed):
    torch.manual_seed(seed)
    cfg = GPT2Config(vocab_size=len(TOK), n_positions=16, n_embd=64, n_layer=2, n_head=4,
                     bos_token_id=0, eos_token_id=V['<eos>'], pad_token_id=0)
    return AutoModelForCausalLM.from_config(cfg)

def eval_policy(model, n=384, seed=123):
    """Vraie récompense moyenne de la politique, échantillonnée sans masque."""
    g = np.random.default_rng(seed)
    tot = 0.0
    for _ in range(n):
        pid = int(g.integers(0, 2))
        ids = [pid]
        for _ in range(LEN_R + 2):
            with torch.no_grad():
                logits = model(torch.tensor([ids])).logits[0, -1]
            ids.append(int(torch.multinomial(F.softmax(logits, -1), 1)))
        content = [t for t in ids if 2 <= t <= 9][:LEN_R]
        content = content + [V['h']] * (LEN_R - len(content))
        tot += true_reward(np.array(content, dtype=np.int64), pid)
    return tot / n

torch.manual_seed(0)
pol_test = make_lm(0)
print(f"politique non entraînée : vraie récompense {eval_policy(pol_test):.3f}"
      f"  (le padding 'h' punit les bavardages prompt/EOS)")

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (0) is identical to the `bos_token_id` (0), `eos_token_id` (10), or the `sep_token_id` (None), and your input is not padded.


politique non entraînée : vraie récompense 0.344  (le padding 'h' punit les bavardages prompt/EOS)


**Lecture chiffree — le point zero avant tout entrainement.** `politique non entraînée : vraie récompense 0.344  (le padding 'h' punit les bavardages prompt/EOS)`. Deux informations dans une ligne. D'abord le niveau : 0.344, SOUS le bruit de fond 0.487 — la politique vierge fait pire que le hasard. La cause est dans la parenthese : les sequences sont paddees avec le token `h`, dont le poids dans W_TOK est negatif (la sortie le dit : il "punit"), donc chaque token de padding coute de la recompense — le modele non entraine "bavarde" (emet des tokens de prompt/EOS) et se fait sanctionner. A retenir pour la suite : ce notebook mesure toujours la VRAIE recompense (celle des poids, pas celle du reward model Bradley-Terry) sur ses evaluations — la divergence entre les deux est precisement ce que le multi-seed de la section 6 met en evidence, le juge appris n'etant qu'une approximation bruitee de la vraie.


## 4. Bras maison : PPO-clip avec value net (recette `RLPT-01` transplantée)

La boucle de `RLPT-01`, adaptée au monde TOK : rollouts masqués au contenu, récompense
terminale = score RM, **avantage = $r - V(x)$** où $V$ est un value net appris (même
architectecture que le RM), ratio clip PPO sur les log-vraisemblances de tokens, et
pénalité KL $\beta$ contre la politique de référence (approximation $k_3$, la même
que dans `trl`).

In [5]:
class ValueNet(nn.Module):
    def __init__(self, vs=len(TOK), hid=64):
        super().__init__()
        self.emb = nn.Embedding(vs, hid)
        self.pos = nn.Embedding(1 + LEN_R, hid)
        self.mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, x):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(L))
        return self.mlp(h.mean(1)).squeeze(-1)

def token_logps(model, x):
    out = torch.log_softmax(model(x).logits, dim=-1)
    tgt = x[:, 1:]
    return out[:, :-1, :].gather(-1, tgt.unsqueeze(-1)).squeeze(-1)   # (B, L-1)

def rollout(policy, n, gen):
    """n trajectoires : prompt + 8 tokens échantillonnés, masqués au contenu."""
    seqs, pids = [], []
    for _ in range(n):
        pid = int(gen.integers(0, 2))
        ids = [pid]
        for _ in range(LEN_R):
            with torch.no_grad():
                logits = policy(torch.tensor([ids])).logits[0, -1]
            mask = torch.full((len(TOK),), -1e9)
            mask[2:10] = 0.0
            ids.append(int(torch.multinomial(F.softmax(logits + mask, -1), 1)))
        seqs.append(ids[1:]); pids.append(pid)
    return seqs, pids

def seqs_to_x(seqs, pids):
    B = len(seqs)
    x = torch.zeros(B, 1 + LEN_R, dtype=torch.long)
    for i, (s, p) in enumerate(zip(seqs, pids)):
        x[i, 0] = p
        x[i, 1:] = torch.from_numpy(np.asarray(s, dtype=np.int64))
    return x

BETA_KL, PPO_STEPS, PPO_BS = 0.05, 12, 192    # 12 x 192 = 2304 rollouts

def train_ppo_maison(policy, ref, value, rm, seed,
                     steps=PPO_STEPS, bs=PPO_BS, lr=3e-4, beta=BETA_KL, clip=0.2, vf=0.5):
    opt = torch.optim.Adam(list(policy.parameters()) + list(value.parameters()), lr=lr)
    gen = np.random.default_rng(seed)
    hist = []
    for it in range(steps):
        seqs, pids = rollout(policy, bs, gen)
        x = seqs_to_x(seqs, pids)
        with torch.no_grad():
            r = rm(x)
            v = value(x)
            adv = (r - v)
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)
            old_lp = token_logps(policy, x)
            ref_lp = token_logps(ref, x)
        lp = token_logps(policy, x)
        ratio = torch.exp((lp - old_lp).sum(1))
        pg = torch.minimum(ratio * adv, torch.clamp(ratio, 1 - clip, 1 + clip) * adv).mean()
        kl = (torch.exp(ref_lp - lp) - (ref_lp - lp) - 1).sum(1).mean()
        vloss = F.mse_loss(value(x), r)
        loss = -pg + vf * vloss + beta * kl
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(dict(r=float(r.detach().mean()), kl=float(kl.detach()),
                         vloss=float(vloss.detach())))
    return hist

In [6]:
t0 = time.time()
torch.manual_seed(0)
pol_m, ref_m = make_lm(0), make_lm(0)
value_m = ValueNet()
r_avant = eval_policy(pol_m)
hist_m = train_ppo_maison(pol_m, ref_m, value_m, RM, seed=0)
t_maison = time.time() - t0
r_apres = eval_policy(pol_m)
print(f"[Bras maison | PPO clip + value net]  {t_maison:.1f}s")
print(f"vraie récompense : {r_avant:.3f} -> {r_apres:.3f}  (bruit de fond {fond:.3f})")
for h in hist_m[::4]:
    print(f"  it r_rm={h['r']:+.3f} kl={h['kl']:.3f} vloss={h['vloss']:.3f}")

[Bras maison | PPO clip + value net]  126.6s
vraie récompense : 0.219 -> 0.928  (bruit de fond 0.487)
  it r_rm=-0.596 kl=0.042 vloss=1.053
  it r_rm=+0.062 kl=0.473 vloss=0.855
  it r_rm=+0.316 kl=0.874 vloss=0.830


**Lecture du bras maison.** Le value net met quelques itérations à caler (la `vloss`
monte d'abord : $V$ apprend pendant que la politique bouge), puis l'avantage
$r - V$ devient informative et la récompense RM décolle. C'est **le** coût caché
du PPO : un critic à entraîner en même temps que la politique, sur un signal
non stationnaire (la cible de $V$ change à chaque update). À l'échelle réelle,
ce critic coûte un modèle entier ; ici 5 505 paramètres suffisent.

Les trois lignes du run mesurent la montée : it r_rm=-0.596 kl=0.042 vloss=1.053 puis it r_rm=+0.062 kl=0.473 vloss=0.855 puis it r_rm=+0.316 kl=0.874 vloss=0.830 — la
récompense RM passe de négative à positive, la `vloss` monte de 1.053 à 0.830
(le critic suit une cible qui bouge), et le gain final en **vraie** récompense va de
0.219 à 0.928 en 126.6 s.


## 5. Bras SOTA : `trl.GRPOTrainer`

Trois différences d'ingénierie avec le bras maison, toutes instructives :

1. **Le reward est une fonction** — `reward_funcs` reçoit `(prompts, completions,
   completion_ids)` en **arguments nommés** et rend une liste de floats. Le juge
   peut être n'importe quoi : ici notre RM, ailleurs un vérificateur Z3, un
   classifieur, une heuristique. C'est l'interface qui a rendu GRPO populaire
   (RLVR — rewards vérifiables).
2. **La baseline est dans la structure** : `num_generations=8` = 8 réponses par
   prompt, avantage normalisé dans le groupe. Pas de value net.
3. **`beta > 0` exige un modèle de référence identifiable** : avec un LM construit
   `from_config` (sans path), trl essaie de recharger un ref depuis le hub et
   échoue (`Repo id must use...`). Le workaround propre : `save_pretrained` vers
   un dossier temporaire et passer le **chemin**.

In [7]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

N_PROMPTS, G = 96, 8          # 96 x 8 x 3 époques = 2304 completions (budget PPO)

def reward_rm(prompts, completions, completion_ids=None, **kw):
    """Score du juge RM sur les complétions -- même filtre contenu que eval_policy."""
    out = []
    for p, ids in zip(prompts, completion_ids):
        pid = V[p.strip()]
        content = [t for t in ids if 2 <= t <= 9][:LEN_R]
        content = content + [V['h']] * (LEN_R - len(content))
        x = torch.zeros(1, 1 + LEN_R, dtype=torch.long)
        x[0, 0] = pid
        x[0, 1:] = torch.from_numpy(np.asarray(content, dtype=np.int64))
        with torch.no_grad():
            out.append(float(RM_SEED_LOCAL(x)[0]))
    return out

def train_grpo_trl(seed, n_epochs=3):
    pol = make_lm(seed)
    pol_dir = tempfile.mkdtemp(prefix='rlpt0f_pol_')
    pol.save_pretrained(pol_dir)          # workaround : beta>0 veut un ref par chemin
    rng = np.random.default_rng(seed)
    ds = Dataset.from_list(
        [{"prompt": TOK[int(rng.integers(0, 2))] + ' '} for _ in range(N_PROMPTS)])
    cfg = GRPOConfig(
        output_dir=tempfile.mkdtemp(prefix='rlpt0e_grpo_'),   # hors du dépôt
        beta=BETA_KL,                      # même coefficient KL que le bras maison
        num_generations=G,
        max_completion_length=LEN_R + 1,
        per_device_train_batch_size=32,
        num_train_epochs=n_epochs,
        learning_rate=1e-4,
        logging_steps=8,
        report_to=[], save_strategy='no', disable_tqdm=True,
        seed=seed, use_cpu=True,
    )
    tr = GRPOTrainer(model=pol_dir, reward_funcs=reward_rm, args=cfg,
                     train_dataset=ds, processing_class=tz)
    tr.train()
    return tr

Budget apparié : le bras maison fait 12 itérations × 192 rollouts = **2 304
réponses évaluées** ; le bras GRPO fait 96 prompts × 8 générations × 3 époques =
**2 304 réponses**. Même juge, même budget d'échantillons, même $\beta$ — les
hyperparamètres restants (lr, clip) sont ceux de chaque méthode par défaut au jouet.

In [8]:
RM_SEED_LOCAL = RM     # le juge global sert au run seed 0 ; run_seed repose le sien

t0 = time.time()
torch.manual_seed(0)
tr0 = train_grpo_trl(0)
t_trl = time.time() - t0
r_trl = eval_policy(tr0.model)
print(f"[Bras SOTA | trl.GRPOTrainer]  {t_trl:.1f}s")
print(f"vraie récompense : {r_trl:.3f}  (maison {r_apres:.3f}, bruit de fond {fond:.3f})")
last = tr0.state.log_history[-2]
print(f"log trl : reward={last.get('reward', float('nan')):.3f} kl={last.get('kl', float('nan')):.3f}"
      f" frac_zero_std={last.get('frac_reward_zero_std', float('nan')):.2f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


...

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


{'loss': '-0.1654', 'grad_norm': '0.7696', 'learning_rate': '1.389e-06', 'num_tokens': '1.979e+04', 'completions/mean_length': '7.977', 'completions/min_length': '1.25', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6914', 'completions/mean_terminated_length': '5.658', 'completions/min_terminated_length': '1.25', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.6444', 'rewards/reward_rm/std': '1.273', 'reward': '-0.6444', 'reward_std': '1.273', 'frac_reward_zero_std': '0', 'kl': '0.08439', 'entropy': '2.31', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.5658', 'epoch': '3'}
{'train_runtime': '39.81', 'train_samples_per_second': '7.234', 'train_steps_per_second': '1.809', 'train_loss': '-0.2186', 'epoch': '3'}


[Bras SOTA | trl.GRPOTrainer]  40.5s
vraie récompense : 0.914  (maison 0.928, bruit de fond 0.487)
log trl : reward=-0.644 kl=0.084 frac_zero_std=0.00


**Lecture chiffree — le premier log du bras SOTA, ligne par ligne.** `[Bras SOTA | trl.GRPOTrainer] 40.5 s`, puis `vraie recompense : 0.914` (maison 0.928, bruit de fond 0.487), puis `log trl : reward=-0.644 kl=0.084 frac_zero_std=0.00`. Trois faits saillants. (1) Le reward du juge au premier log est négatif (-0.644) : le juge Bradley-Terry est severe et bruite, la normalisation par groupe a du grain a moudre. (2) `frac_zero_std=0.00` — aucun groupe n'a de récompense constante (a comparer avec les groupes a variance nulle du GRPO de `RLPT-02` : ici le reward continu evite les groupes plats). (3) `kl=0.084` : la lib rapporte le meme coefficient que les autres bras, donc la comparaison se fait a budget de rollouts et de regularisation egaux. Ce qui compte pour la suite est la **vraie** recompense, 0.914 — pas le score du juge.


## 5bis. Le bras exilé : `trl.experimental.ppo.PPOTrainer`

L'espace racine de `trl` n'expose plus `PPOTrainer`, et l'espace `trl.experimental.ppo`
qui l'abritait n'est **plus livré** : la surface a été retirée du paquet en `trl`
1.13.0 (présente dans les wheels 1.10.0, 1.11.0 et 1.12.0). Le trainer n'a pas été
« supprimé » au sens de la lib — il a été retiré de la distribution. Ce que cette
section conserve est donc **le code** du geste, tel que la lib l'attendait : on remet à
la lib la politique, un **reward model au format lib** (backbone + tête `.score()`,
l'interface `AutoModelForSequenceClassification`), un **value model** du même format (le
critic, appris en ligne), et elle tient la boucle **complète** — rollout `max_new_tokens`,
score du juge, **GAE** ($\gamma$, $\lambda$), ratio clippé policy **et** value, pénalité
KL par token. C'est le PPO complet que le bras maison implémentait à la main, rendu ici
par la lib.

Deux conséquences d'ingénierie, instructives elles aussi :

1. **Le juge change de format, pas de recette.** `get_reward` évalue la séquence **brute**
(prompt + réponse telle que générée, les `<eos>` masqués par l'attention, score relevé au
token qui précède le premier `<eos>`). On réentraîne donc un juge Bradley-Terry de la même
recette que la section 2 — mêmes paires dérivées de la vraie récompense avec écart minimal,
même perte BT — mais en `GPT2ForSequenceClassification`, sur des réponses brutes au lieu
du contenu filtré.
2. **Le pad ne peut plus être `<pA>`.** `generate()` reconstruit le masque d'attention par
`queries != pad_token_id` : avec pad = `<pA>` (id 0), toute query `<pA>` serait masquée
entièrement. Ce bras utilise donc un pad `<eos>`, et les `<eos>` générés en milieu de
réponse jouent le rôle de fin de séquence.

Le budget reste apparié : 2 304 prompts × une passe = 2 304 completions, comme les
deux autres bras — chaque prompt est vu une fois, rollouts frais comme le bras maison.

**Sous la version de `trl` installée ici, ce bras ne s'exécute pas** : la cellule 5bis
constate l'absence de la surface et l'affiche, et le tableau final (section 7) compare
les deux bras qui s'exécutent en signalant la colonne manquante.


In [9]:
# 5bis -- etape 1 : le juge au format lib (backbone + tete .score(), la voie exacte de get_reward)
from transformers import AutoModelForSequenceClassification

tz_ppo = make_tz()
tz_ppo.pad_token = '<eos>'     # un pad <pA> (id 0) masquerait les query <pA> dans generate()

def gpt2_cls_config():
    return GPT2Config(vocab_size=len(TOK), n_positions=16, n_embd=64, n_layer=2, n_head=4,
                      num_labels=1, bos_token_id=0, eos_token_id=V['<eos>'], pad_token_id=0)

def conv_get_reward(x):
    """La convention de get_reward avec pad=<eos> : <eos> masques par l'attention, ids mis a 0,
    positions compactees, score releve au token qui precede le premier <eos> (sinon dernier)."""
    eos = (x == V['<eos>'])
    xm, am = x.masked_fill(eos, 0), (~eos).long()
    pos = am.cumsum(1) - am.long()
    resp = eos[:, 1:]
    first = torch.where(resp.any(1), resp.float().argmax(1), torch.full((x.shape[0],), float(LEN_R)))
    return xm, am, pos, first.long()

def rm_pos_scores(rm, x):
    """Score par la voie de la lib : backbone -> .score() -> position de fin effective."""
    xm, am, pos, last = conv_get_reward(x)
    h = rm.transformer(input_ids=xm, attention_mask=am, position_ids=pos,
                       output_hidden_states=True, return_dict=True).hidden_states[-1]
    return rm.score(h).squeeze(-1)[torch.arange(x.shape[0]), last]

def contenu(ids):
    """Reponse brute -> contenu filtre + pad 'h', la convention de eval_policy."""
    c = [t for t in ids if 2 <= t <= 9][:LEN_R]
    return np.asarray(c + [V['h']] * (LEN_R - len(c)))

def make_pairs_raw(n_pairs, rng, min_gap=0.3, beta=1.0):
    """Meme recette BT que make_pairs, sur des reponses BRUTES (tout le vocab) :
    c'est la sequence que get_reward verra. Vraie recompense toujours calculee sur le contenu filtre."""
    pw, y, pr = [], [], []
    while len(pw) < n_pairs:
        prompt = int(rng.integers(0, 2))
        a, b = rng.integers(0, len(TOK), size=LEN_R), rng.integers(0, len(TOK), size=LEN_R)
        ra, rb = true_reward(contenu(a), prompt), true_reward(contenu(b), prompt)
        if abs(ra - rb) < min_gap:
            continue
        p_a = 1.0 / (1.0 + np.exp(-beta * (ra - rb)))
        pw.append((a, b)); y.append(1 if rng.random() < p_a else 0); pr.append(prompt)
    return pw, np.array(y), np.array(pr)

def x_from_raw(raw_list, pr_list):
    x = torch.zeros(len(raw_list), 1 + LEN_R, dtype=torch.long)
    for i, (raw, p) in enumerate(zip(raw_list, pr_list)):
        x[i, 0] = p; x[i, 1:] = torch.from_numpy(np.asarray(raw, dtype=np.int64))
    return x

def make_rm_gpt2(seed, n_pairs=4000, epochs=60, bs=256, lr=1e-3):
    """Juge Bradley-Terry de la recette rlpt_0, en GPT2ForSequenceClassification."""
    torch.manual_seed(seed + 10000)
    rng = np.random.default_rng(seed + 10000)
    pw, y, pr = make_pairs_raw(n_pairs, rng)
    rm = AutoModelForSequenceClassification.from_config(gpt2_cls_config())
    opt = torch.optim.Adam(rm.parameters(), lr=lr)
    xa = x_from_raw([a for a, b in pw], pr); xb = x_from_raw([b for a, b in pw], pr)
    yt = torch.tensor(y, dtype=torch.long)
    for _ in range(epochs):
        perm = torch.randperm(len(y)).tolist()
        for i in range(0, len(y), bs):
            idx = torch.tensor(perm[i:i + bs])
            sa, sb = rm_pos_scores(rm, xa[idx]), rm_pos_scores(rm, xb[idx])
            diff = torch.where(yt[idx] == 1, sa - sb, sb - sa)
            loss = F.binary_cross_entropy_with_logits(diff, torch.ones(len(idx)))
            opt.zero_grad(); loss.backward(); opt.step()
    return rm

t0 = time.time()
RM_GPT2 = make_rm_gpt2(0)
g = np.random.default_rng(31337)
raws = [g.integers(0, len(TOK), size=LEN_R) for _ in range(300)]
pids = [int(g.integers(0, 2)) for _ in range(300)]
with torch.no_grad():
    s_rm = rm_pos_scores(RM_GPT2, x_from_raw(raws, pids)).numpy()
r_vrai = np.array([true_reward(contenu(r_), p) for r_, p in zip(raws, pids)])
print(f"RM-GPT2 (juge du bras historique) entraîné en {time.time()-t0:.1f}s")
print(f"accord du juge avec la vraie récompense (spearman, 300 réponses brutes) :"
      f" {spearmanr(s_rm, r_vrai).statistic:.3f}")


RM-GPT2 (juge du bras historique) entraîné en 169.3s
accord du juge avec la vraie récompense (spearman, 300 réponses brutes) : 0.552


**Lecture chiffrée — le juge du bras historique.** 169.3 s d'entraînement (4000 paires Bradley-Terry, 60 epochs) pour un accord avec la vraie récompense de **spearman 0.552** sur 300 réponses brutes. Le contrat est plus rude que celui du RM custom de la section 2, qui juge le *contenu* filtré des réponses : l'interface `reward_model` de la lib évalue la séquence **telle qu'elle est générée** — pad `<eos>` compris, tout le vocabulaire — et relève le score à la position de fin de réponse détectée par la lib (avant le premier `<eos>`, sinon au dernier token). Ce juge s'entraîne donc et se mesure **même quand le bras qu'il sert ne peut pas s'exécuter** : la conversion de format (section 5bis) est un travail de la lib, pas de la boucle PPO, et il reste instructif sous la version de `trl` installée ici.


In [10]:
# 5bis -- etape 2 : le bras. La lib tient la boucle complete (rollout, score, GAE, PPO-clip
# policy ET value, penalite KL par token) ; on fournit policy, ref, juge, critic.
def train_ppo_trl(seed, n_prompts=2304, n_epochs=1, bs=192, lr=1e-4):
    pol, ref = make_lm(seed), make_lm(seed)
    value = AutoModelForSequenceClassification.from_config(gpt2_cls_config())   # critic from scratch
    rng = np.random.default_rng(seed + 50000)
    ds = Dataset.from_list([{"input_ids": [int(rng.integers(0, 2))]} for _ in range(n_prompts)])
    cfg = PPOConfig(
        output_dir=tempfile.mkdtemp(prefix='rlpt0f_ppo_'),    # hors du dépôt
        per_device_train_batch_size=bs,
        num_train_epochs=n_epochs,         # 2304 prompts x 1 passe = 2304 completions, budget apparié
        learning_rate=lr,                  # même lr que le bras GRPO (même pile lib)
        response_length=LEN_R,
        stop_token_id=None,                # réponse pleine longueur, comme les autres bras
        temperature=1.0,                   # échantillonnage nu, comme le rollout maison
        num_ppo_epochs=1,                  # une passe par rollout, comme le bras maison
        num_mini_batches=4,
        vf_coef=0.5,                       # même poids value que le bras maison
        kl_coef=BETA_KL,                   # même coefficient KL que les deux autres bras
        local_rollout_forward_batch_size=64,
        logging_steps=4,
        report_to=[], save_strategy='no', disable_tqdm=True,
        num_sample_generations=0,
        seed=seed, use_cpu=True,
    )
    tr = PPOTrainer(args=cfg, processing_class=tz_ppo, model=pol, ref_model=ref,
                    reward_model=RM_GPT2, train_dataset=ds, value_model=value)
    tr.train()
    return tr

# Ce bras depend entierement de la surface `trl.experimental.ppo`, retiree du paquet TRL en
# 1.13.0. Le code reste lisible (il documente le geste d'API), mais il ne s'execute que si la
# surface est presente ; sinon l'absence est AFFICHEE, jamais silencieuse.
if not _PPO_AVAILABLE:
    r_avant_ppo = r_ppotrl = t_ppotrl = None
    print("[Bras historique | trl.experimental.ppo.PPOTrainer]  NON EXECUTE -- "
          "surface `trl.experimental.ppo` absente du paquet TRL installe (retiree en 1.13.0)")
else:
    t0 = time.time()
    torch.manual_seed(0)
    r_avant_ppo = eval_policy(make_lm(0))
    tr_ppo = train_ppo_trl(0)
    t_ppotrl = time.time() - t0
    r_ppotrl = eval_policy(tr_ppo.policy_model)
    print(f"[Bras historique | trl.experimental.ppo.PPOTrainer]  {t_ppotrl:.1f}s")
    print(f"vraie récompense : {r_avant_ppo:.3f} -> {r_ppotrl:.3f}  (bruit de fond {fond:.3f})")
    last = [d for d in tr_ppo.state.log_history if 'objective/scores' in d][-1]
    print(f"log trl-ppo : scores={last['objective/scores']:.3f} kl={last['objective/kl']:.3f}"
          f" vf_loss={last['loss/value_avg']:.3f} eps={last['eps']}")


[Bras historique | trl.experimental.ppo.PPOTrainer]  NON EXECUTE -- surface `trl.experimental.ppo` absente du paquet TRL installe (retiree en 1.13.0)


**Le bras historique, non exécuté.** La cellule 5bis affiche l'absence au lieu de la contourner : la surface `trl.experimental.ppo` n'est plus livrée par `trl` depuis la 1.13.0 (présente dans les wheels 1.10.0, 1.11.0 et 1.12.0, absente à partir de 1.13.0 — fenêtre mesurée par inspection des wheels PyPI). Le code du bras est conservé tel quel dans la cellule : il documente le contrat d'API (`reward_model` au format de la lib, value model `GPT2ForSequenceClassification`, `PPOConfig` avec `response_length`, `num_ppo_epochs`, `vf_coef`, `kl_coef`), et il s'exécute chez qui installe `trl==1.12.0`. **Les chiffres de ce bras ne figurent donc plus dans les sorties committées** : ceux qui figuraient dans les versions précédentes du carnet ont été produits sous `trl` 1.9.2 et restent consultables dans l'historique du fichier. Ce que le carnet en retient, c'est la leçon : **une comparaison SOTA se date par la version de la bibliothèque**, pas seulement par celle du noyau — et le tableau final (section 7) affiche une colonne « non disponible » plutôt qu'un chiffre périmé.


## 6. Multi-seed {0, 1, 7, 42}

Chaque seed **retire tout le pipeline** : nouvelles paires, nouveau RM, nouvelles
politiques. C'est la force du monde synthétique — la variabilité du juge lui-même
entre dans la mesure.

In [11]:
# Multi-seed : on coupe le bruit de lib (barres tqdm, warnings transformers) --
# les runs detailles avec leurs logs vivent deja en sections 4, 5 et 5bis.
import io, contextlib
import transformers as _tfm
_tfm.logging.set_verbosity_error()
_tfm.logging.disable_progress_bar()

def run_seed(seed):
    global RM_SEED_LOCAL
    torch.manual_seed(seed)
    RM_SEED_LOCAL = make_rm(seed)
    out = {}

    t0 = time.time()
    pol, ref = make_lm(seed), make_lm(seed)
    value = ValueNet()
    r0 = eval_policy(pol, seed=seed * 1000 + 123)
    with contextlib.redirect_stdout(io.StringIO()):  # logs par update maison : captures
        train_ppo_maison(pol, ref, value, RM_SEED_LOCAL, seed=seed)
    out['maison'] = dict(r=eval_policy(pol, seed=seed * 1000 + 123), t=time.time() - t0, r0=r0)

    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):  # dicts de log GRPO : captures
        tr = train_grpo_trl(seed)
    out['trl'] = dict(r=eval_policy(tr.model, seed=seed * 1000 + 123), t=time.time() - t0, r0=r0)
    return out

RES = {}
for s in SEEDS[:1]:
    RES[s] = run_seed(s)
    m, c = RES[s]['maison'], RES[s]['trl']
    print(f"seed {s}: maison {m['r0']:.3f}->{m['r']:.3f} ({m['t']:.0f}s) | "
          f"trl {c['r0']:.3f}->{c['r']:.3f} ({c['t']:.0f}s)")
print("(les seeds restantes tournent -- cellule suivante)")


seed 0: maison 0.219->0.928 (125s) | trl 0.219->0.914 (60s)
(les seeds restantes tournent -- cellule suivante)


**Ce que `run_seed` rejoue, et pourquoi c'est la vraie robustesse.** La fonction relance TOUT le pipeline par graine : nouvelles paires de preferences, nouveau reward model (RM_SEED_LOCAL repose le sien), nouvelles politiques initiales — les barres `Writing model shards` et `Loading weights` (0/28) le rappellent a chaque relance. La consequence methodologique est la difference silencieuse de ce notebook avec un multi-seed ordinaire : la variabilite measuree a la fin inclut celle du JUGE lui-meme, pas seulement celle de l'optimisation. Deux juges differents peuvent classer differemment les memes reponses ; un gain qui tient sur 4 seeds a juges independants est un gain qui survit a l'incertitude du juge — exactement la propriete qu'un reward model appris doit avoir en production, et que le bruitage Bradley-Terry des paires d'entrainement simule ici a petite echelle.


In [12]:
for s in SEEDS[1:]:
    RES[s] = run_seed(s)
    m, c = RES[s]['maison'], RES[s]['trl']
    print(f"seed {s}: maison {m['r0']:.3f}->{m['r']:.3f} ({m['t']:.0f}s) | "
          f"trl {c['r0']:.3f}->{c['r']:.3f} ({c['t']:.0f}s)")

gains_m = np.array([RES[s]['maison']['r'] - RES[s]['maison']['r0'] for s in SEEDS])
gains_c = np.array([RES[s]['trl']['r'] - RES[s]['trl']['r0'] for s in SEEDS])
ts_m = np.array([RES[s]['maison']['t'] for s in SEEDS])
ts_c = np.array([RES[s]['trl']['t'] for s in SEEDS])
print(f"\ngain maison : {gains_m.mean():+.3f} ± {gains_m.std():.3f}  (temps moyen {ts_m.mean():.0f}s)")
print(f"gain trl    : {gains_c.mean():+.3f} ± {gains_c.std():.3f}  (temps moyen {ts_c.mean():.0f}s)")
diff = gains_c.mean() - gains_m.mean()
sig = max(gains_m.std(), gains_c.std())
print(f"écart trl-maison : {diff:+.3f} (2σ = {2*sig:.3f}) -> "
      + ("INCONCLUSIVE" if abs(diff) < 2 * sig else f"{'trl' if diff > 0 else 'maison'} devant"))

seed 1: maison 0.212->0.716 (107s) | trl 0.212->0.777 (78s)


seed 7: maison 0.227->0.935 (152s) | trl 0.227->0.916 (76s)


seed 42: maison 0.341->0.964 (140s) | trl 0.341->1.041 (78s)

gain maison : +0.636 ± 0.084  (temps moyen 131s)
gain trl    : +0.662 ± 0.056  (temps moyen 73s)
écart trl-maison : +0.026 (2σ = 0.167) -> INCONCLUSIVE


**Lecture chiffrée — les quatre seeds côte à côte, et la leçon de dispersion.** `seed 0: maison 0.219->0.928 (125s) | trl 0.219->0.914 (60s)`

`seed 1: maison 0.212->0.716 (107s) | trl 0.212->0.777 (78s)`

`seed 7: maison 0.227->0.935 (152s) | trl 0.227->0.916 (76s)`

`seed 42: maison 0.341->0.964 (140s) | trl 0.341->1.041 (78s)` Puis le résumé : `gain maison : +0.636 ± 0.084  (temps moyen 131 s)`, `gain trl    : +0.662 ± 0.056  (temps moyen 73 s)` et `écart trl-maison : +0.026 (2σ = 0.167) -> INCONCLUSIVE`. La lecture honnête : l'écart moyen (+0.026) reste du même ordre que la dispersion des graines (|0.026| contre 2σ = 0.167), le verdict reste donc **INCONCLUSIVE** : l'écart ne se distingue pas du bruit de graine — mais la **dispersion** est le résultat visible à l'œil nu dans les quatre lignes : l'écart-type du bras `trl` (0.056) est plus faible que celui du bras maison (0.084) — le bras `trl` est plus régulier d'une graine à l'autre La régularité d'un bras contre la volatilité de l'autre : c'est la conclusion, et le INCONCLUSIVE porte dessus — cet écart de régularité, lui, ne demande pas de test statistique pour être lu.


## 7. From scratch vs SOTA : le tableau

Lignes de code comptées sur les sources du notebook livré (lignes non vides des
définitions, docstrings incluses) : bras maison = ValueNet + token_logps + rollout +
seqs_to_x + train_ppo_maison = 60 ; bras trl = reward_rm + train_grpo_trl (incluant
le bloc GRPOConfig et la construction du dataset) = 36 ; bras historique = juge au
format lib (contenu, conv_get_reward, rm_pos_scores, make_pairs_raw, x_from_raw,
make_rm_gpt2) + train_ppo_trl = 85 — même convention (le juge compte dans le bras,
comme reward_rm compte dans le bras GRPO ; le RM custom de la section 2, lui, est
commun aux deux premiers bras et compté nulle part).


In [13]:
# LOC pré-mesurées sur les sources du notebook livré (lignes non vides, docstrings incluses)
loc_m = 60
loc_c = 36
loc_p = 85

if _PPO_AVAILABLE:
    col_gain = f"{r_ppotrl - r_avant_ppo:>19.3f}"
    col_std = f"{'(seed 0 seul)':>19}"
    col_t = f"{t_ppotrl:>18.0f}s"
    col_dep = f"{'+trl (experimental)':>19}"
    note = "NB : PPO-trl mesuré au seed 0 uniquement (section 5bis) ; maison/GRPO sur 4 seeds (section 6)"
else:
    col_gain = f"{'non disponible':>19}"
    col_std = f"{'surface retiree':>19}"
    col_t = f"{'-':>18}"
    col_dep = f"{'surface retiree':>19}"
    note = ("NB : le bras `trl.experimental.ppo.PPOTrainer` n'a pas ete execute -- la surface a ete "
            "retiree du paquet TRL en 1.13.0 ; maison/GRPO sur 4 seeds (section 6)")

print(f"{'':<24}{'PPO maison':>19}{'trl.GRPOTrainer':>19}{'trl.PPOTrainer':>19}")
print(f"{'gain vraie récomp.':<24}{gains_m.mean():>19.3f}{gains_c.mean():>19.3f}{col_gain}")
print(f"{'dispersion (std)':<24}{gains_m.std():>19.3f}{gains_c.std():>19.3f}{col_std}")
print(f"{'temps moyen / run':<24}{ts_m.mean():>18.0f}s{ts_c.mean():>18.0f}s{col_t}")
print(f"{'lignes de code':<24}{loc_m:>19}{loc_c:>19}{loc_p:>19}")
print(f"{'critic':<24}{'value net (appris)':>19}{'groupe (structural)':>19}{'value head + GAE':>19}")
print(f"{'dépendances':<24}{'torch':>19}{'+trl, datasets':>19}{col_dep}")
print(note)


                                 PPO maison    trl.GRPOTrainer     trl.PPOTrainer
gain vraie récomp.                    0.636              0.662     non disponible
dispersion (std)                      0.084              0.056    surface retiree
temps moyen / run                      131s                73s                 -
lignes de code                           60                 36                 85
critic                   value net (appris)groupe (structural)   value head + GAE
dépendances                           torch     +trl, datasets    surface retiree
NB : le bras `trl.experimental.ppo.PPOTrainer` n'a pas ete execute -- la surface a ete retiree du paquet TRL en 1.13.0 ; maison/GRPO sur 4 seeds (section 6)


**Lecture chiffrée — le tableau final, ligne par ligne.** `gain vraie récomp.` — +0.636 / +0.662 / non disponible : les deux bras qui s'exécutent apprennent, et le troisième affiche l'absence de la surface `trl.experimental.ppo` (retirée du paquet en `trl` 1.13.0) au lieu d'un chiffre produit sous une version de `trl` où elle existait ; `dispersion (std)` — 0.084 contre 0.056 : l'écart-type du bras `trl` (0.056) est plus faible que celui du bras maison (0.084) — le bras `trl` est plus régulier d'une graine à l'autre ; `temps moyen / run` — 131 s / 73 s / - : le bras maison est 1.8× plus lent que le bras `trl` ; `lignes de code` — 60 / 36 / 85 : le paradoxe du bras historique, son cœur `train_ppo_trl` est le plus court des trois, mais l'interface `reward_model` au format de la lib impose de reconstruire le juge — qui pèse à lui seul plus du double du cœur du trainer — : l'interface est le produit, pas le trainer ; `critic` — value net appris / moyenne de groupe structurelle / value head + GAE : trois réponses distinctes à la question de la baseline ; `dépendances` — torch seul / + trl, datasets / surface retirée. C'est le contrat du from scratch rendu visible en six lignes, avec le INCONCLUSIVE de la section 6 en filigrane : trancher maison contre GRPO demanderait un multi-seed plus profond — et le bras historique, désormais non exécutable sous la version installée, mériterait le même traitement **avant** tout verdict, chez qui installe `trl==1.12.0`.


**Pourquoi du from scratch.** La boucle PPO maison rend visibles les trois pièces que
GRPO factorise : le critic (vous l'entraînez, vous voyez sa `vloss` dériver quand la
politique bouge), le clip (vous pouvez le désactiver et regarder exploser), la
pénalité KL (même $\beta$ des deux côtés — comparez les colonnes `kl`). Quand le bras
trl et le bras maison arrivent au même niveau, ce n'est pas une coïncidence : c'est
le même théorème de politique améliorée, deux estimateurs de l'avantage.

**Quand le SOTA.** `GRPOTrainer` n'existe pas pour battre une boucle sur
un monde à 11 tokens — il existe parce que la même interface accepte demain Qwen à 7B,
vLLM, LoRA, la distribution multi-GPU, et surtout **des rewards vérifiables** : la
signature `reward_funcs(prompts, completions, completion_ids)` a fait de GRPO le
moteur du RLVR. Le from scratch apprend **ce que fait l'option** ; le SOTA est **le
moyen de ne pas l'écrire soi-même** en production.

**Et le PPO dans tout ça ?** Il n'a pas disparu du champ théorique — c'est toujours
le socle (GRPO = PPO sans critic, baseline déplacée dans le groupe). Il a disparu du
**catalogue `trl`**, et c'est l'enseignement transversal de ce notebook : une
comparaison « SOTA » est datée par construction, et datée **par la bibliothèque** avant
de l'être par le noyau. `RLPT-01` a appris PPO ; `RLPT-00e` a appris DPO ; ici le SOTA
en ligne s'appelle GRPO. La pile bouge, la mathématique de la baseline reste.


## 8. Ce qu'il faut retenir

| Leçon | Mesure dans ce notebook |
|---|---|
| La baseline est LE choix de design | value net appris (PPO) vs moyenne de groupe (GRPO) — même récompense terminale, même budget |
| RLHF optimise le juge, pas le vrai | le gain se mesure en **vraie** récompense, jamais en score RM |
| GRPO = PPO sans critic | même budget, même famille d'avantage, zéro paramètre de value |
| Une surface SOTA est **datée** | `trl.experimental.ppo` livrée de `trl` 1.10.0 à 1.12.0, **retirée du paquet en 1.13.0** — la cellule d'import **constate** l'absence au lieu de la supposer |
| L'interface reward est le produit | `reward_funcs(prompts, completions, completion_ids)` côté GRPO, `reward_model` au format lib côté PPO : le juge épouse la pile |

**Ce que ce notebook ajoute à la série** : `RLPT-01` écrivait PPO from scratch sur sa
propre tâche ; `RLPT-00e` branchait le SOTA **offline** (DPO) sur le monde de `RLPT-00`.
Ici la boucle fermée **juge → échantillonnage → politique** passe en ligne, le SOTA
de l'époque (`GRPOTrainer`) est départagé contre la recette maison à budget exact de
rollouts, et le sort de `PPOTrainer` — le sujet initial de l'issue — est mesuré :
la surface a existé, elle a été retirée du paquet, et le carnet le **constate** au lieu
de le faire semblant (section 5bis).


## Exercices

Trois exercices, du plus guidé au plus ouvert. Ils étendent les fonctions définies
ci-dessus — aucune nouvelle dépendance.

In [14]:
# Exercice 1 -- Le coefficient KL : balayer beta
# TODO etudiant
# Etape 1 : re-entraîner le bras maison pour beta dans [0.0, 0.01, 0.05, 0.2]
#            (train_ppo_maison(pol, ref, value, RM, seed=0, beta=...))
# Etape 2 : pour chaque beta, mesurer (a) le gain en vraie récompense, (b) la KL finale
#            contre ref -- la KL est deja dans hist (dernier element : hist[-1]['kl'])
# Etape 3 : tracer le compromis gain vs KL. Ou se situe le "coude" ?
resultats_beta = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


In [15]:
# Exercice 2 -- RLOO : la baseline leave-one-out, a la main
# TODO etudiant
# Etape 1 : copier rollout/seqs_to_x ; generer G=8 reponses par prompt (meme prompt G fois)
# Etape 2 : avantage RLOO de la reponse i : A_i = r_i - mean(r_j, j != i)
#            (la baseline de i ignore i -- c'est tout le "leave-one-out")
# Etape 3 : re-utiliser la boucle PPO (clip + KL) avec ce nouvel avantage, sans value net
# Etape 4 : comparer a PPO et a GRPO sur la meme metrique -- RLOO est le 3e estimateur
resultats_rloo = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


In [16]:
# Exercice 3 -- Reward hacking : un juge biaise par la longueur
# TODO etudiant
# Etape 1 : construire un RM biaise : re-entraîner RewardModel sur des paires dont le
#            gagnant est artificiellement allonge (label flippe si len(gagnant) > len(perdant))
# Etape 2 : entraîner la politique maison contre CE juge (meme boucle, rm=biasé)
# Etape 3 : mesurer l'ecart : score du juge biaise vs VRAIE récompense de la politique
#            -- la politique exploite-t-elle le biais ? (c'est rlpt_3, mais en ligne)
resultats_hack = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


## Références

- `RLPT-00` — le monde synthétique et le reward model Bradley-Terry (même dépôt).
- `RLPT-01` — PPO from scratch pour l'alignement d'un LM (même dépôt).
- `RLPT-00e` — le geste SOTA offline : `trl.DPOTrainer` sur le même monde (même dépôt).
- `RLPT-02` / `PT-11a` — GRPO respectivement à la main sur Qwen et à l'échelle avec
  `trl.GRPOTrainer` + rewards vérifiables (même dépôt).
- Shao et al. 2024, *DeepSeekMath* — la section qui introduit GRPO (group relative
  policy optimization).
- Schulman et al. 2017, *Proximal Policy Optimization* — le clip et le critic que
  GRPO supprime.
- Rafailov et al. 2023, *Direct Preference Optimization* — la voie offline de `RLPT-00e`.
- Documentation TRL — l'entrée `ppo_trainer` a disparu de la documentation avec la
  surface : `trl` 1.13.0 retire `trl/experimental/ppo/` du paquet (fenêtre de
  disponibilité mesurée : `trl` 1.10.0 à 1.12.0).


**Navigation** : Suivant → [RLPT-00g -- PPO avec GAE, et le retrait de la surface `trl.experimental.ppo`](RLPT-00g-PPO-TRL-Experimental-Python.ipynb) : meme monde synthetique, meme juge BT, troisieme implementation du geste online -- la PPO avec GAE y est une **copie pedagogique declaree**, la surface de la librairie n'etant plus livree depuis `trl` 1.13.0.
